# Day 43: Lab - House Price Prediction Mini-Project

## Project Goal
This lab pulls together everything from the past few days - regression, train/test splitting, and regularization - into one small end-to-end workflow, which mirrors how a real regression project is structured: get data, split it, train candidate models, then evaluate and compare them. We use a synthetic housing dataset instead of a real one so the underlying relationship between features (size, bedrooms, age, distance to city) and price is fully known and controllable, which makes it easier to sanity-check that the models are learning something sensible. The cell below builds this dataset by generating random feature values and then computing `price` from a fixed formula plus random noise, so we know in advance which features should matter and roughly how much.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 300
df = pd.DataFrame({
    'size_sqft': np.random.randint(600, 3500, n),
    'bedrooms': np.random.randint(1, 6, n),
    'age_years': np.random.randint(0, 40, n),
    'distance_to_city_km': np.round(np.random.uniform(1, 30, n), 1)
})
# construct price as a known linear combination of features plus noise, so we can later
# check whether the trained models recover a sensible relationship
df['price'] = (
    df['size_sqft'] * 120 + df['bedrooms'] * 8000 - df['age_years'] * 500
    - df['distance_to_city_km'] * 300 + np.random.normal(0, 8000, n)
)
print(df.head())

## Step 1: Train-Test Split
Before training anything, we set aside a portion of the data (the test set) that the model never sees during training. This is essential because evaluating a model on the same data it was trained on gives an overly optimistic, misleading picture of how well it actually generalizes - a model can memorize training data without learning the true underlying pattern. Here `X` holds all the feature columns (everything except price) and `y` holds the target we want to predict. Using `test_size=0.2` reserves 20% of the houses for testing, and `random_state=42` makes the split reproducible so we get the same train/test rows every time this notebook runs.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price'])  # features: everything except the target column
y = df['price']  # target: what we're trying to predict
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 2: Train Models
It's good practice to compare a plain baseline model against a regularized alternative rather than assuming one is automatically better - the right choice depends on the data. Here we train an ordinary `LinearRegression` (no penalty on coefficients, fits training data as closely as possible) alongside a `Ridge` regression with `alpha=10` (a fairly strong L2 penalty that shrinks coefficients to reduce overfitting and improve stability, especially useful if features like size and bedrooms are correlated). Both models are fit only on `X_train`/`y_train` - the test set stays untouched until evaluation. Neither line prints anything; the trained model objects (`linear`, `ridge`) are what we'll use for predictions in the next step.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

linear = LinearRegression().fit(X_train, y_train)  # unregularized baseline
ridge = Ridge(alpha=10).fit(X_train, y_train)  # L2-regularized alternative for comparison

## Step 3: Evaluate & Compare
The final step of any modeling workflow is to score each candidate model on the held-out test set and pick the best one using consistent metrics. We use two complementary metrics here: R2 (the proportion of variance in price explained by the model - closer to 1 means a better fit, 0 means no better than predicting the mean) and MAE, Mean Absolute Error (the average dollar amount the predictions are off by, in the same units as price, which is easier to explain to a non-technical audience than R2). Looking at both together avoids being misled by just one number - a model could have a decent R2 but still be off by a large dollar amount on individual houses. When you read the printed results, prefer the model with higher R2 and lower MAE; if Ridge and Linear score very similarly, it suggests overfitting wasn't a big problem here and the extra regularization from Ridge isn't buying much.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

for name, m in [('Linear', linear), ('Ridge', ridge)]:
    preds = m.predict(X_test)  # predictions on the untouched test set
    print(name, 'R2:', round(r2_score(y_test, preds), 3), 'MAE:', round(mean_absolute_error(y_test, preds), 2))